In [1]:
import torch
from gpt_model import GPTModel
from gpt_config import GPT_CONFIG_124M, GPT_CONFIG_MEDIUM
import tiktoken

In [2]:
tokenizer = tiktoken.get_encoding("gpt2")

In [3]:
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)

In [4]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
out = model(batch)

In [5]:
print("input batch: \n", batch)
print("output shape: ", out.shape)
print(out)

input batch: 
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])
output shape:  torch.Size([2, 4, 50257])
tensor([[[ 0.3613,  0.4222, -0.0711,  ...,  0.3483,  0.4661, -0.2838],
         [-0.1792, -0.5660, -0.9485,  ...,  0.0477,  0.5181, -0.3168],
         [ 0.7120,  0.0332,  0.1085,  ...,  0.1018, -0.4327, -0.2553],
         [-1.0076,  0.3418, -0.1190,  ...,  0.7195,  0.4023,  0.0532]],

        [[-0.2564,  0.0900,  0.0335,  ...,  0.2659,  0.4454, -0.6806],
         [ 0.1230,  0.3653, -0.2074,  ...,  0.7705,  0.2710,  0.2246],
         [ 1.0558,  1.0318, -0.2800,  ...,  0.6936,  0.3205, -0.3178],
         [-0.1565,  0.3926,  0.3288,  ...,  1.2630, -0.1858,  0.0388]]],
       grad_fn=<UnsafeViewBackward0>)


In [6]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 163,009,536


In [7]:
print("token embeddings: ", model.tok_emb.weight.shape)
print("out heads: ", model.out_head.weight.shape)

token embeddings:  torch.Size([50257, 768])
out heads:  torch.Size([50257, 768])


In [8]:
total_size_bytes = total_params * 4
total_size_mb = total_size_bytes / 1024 / 1024
print(f"Total size of model in MB: {total_size_mb:.2f}")

Total size of model in MB: 621.83


In [9]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        ids_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(ids_cond)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=-1)
    return idx

In [10]:
start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded ID: ", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape: ", encoded_tensor.shape)

encoded ID:  [15496, 11, 314, 716]
encoded_tensor.shape:  torch.Size([1, 4])


In [11]:
model.eval()
out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"]
)
print("output: ", out)
print("output length: ", len(out[0]))

output:  tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267]])
output length:  10


In [12]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print("decoded text: ", decoded_text)

decoded text:  Hello, I am Featureiman Byeswickattribute argue
